### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="japanese_credit_screening",
    dataset_year="1992",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5259N",
    download_description="""
We get the credit screening data from UCI.

wget https://archive.ics.uci.edu/static/public/28/japanese+credit+screening.zip && unzip japanese+credit+screening.zip credit.lisp && rm japanese+credit+screening.zip && mkdir -p local-data-warehouse/japanese_credit_screening && mv credit.lisp  local-data-warehouse/japanese_credit_screening/
""",
    # References
    academic_reference_bibtex="""@misc{Sano1992japaneseCreditScreening,
  author       = {Sano, Chiharu},
  title        = {{Japanese Credit Screening}},
  year         = {1992},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C5259N}
}
""",
    academic_reference_bibtex_key="Sano1992japaneseCreditScreening",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We read the data from the .lisp file.

- We treat the `credit_screening` predicate as the target variable, which is binary (1 for positive, 0 for negative).
- We merge female and male indicators into a single `gender` categorical feature.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="credit_screening",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="credit_screening",
)

## Preprocessing

In [2]:
# Parse data from .lisp file (thanks to ChatGPT)
import re
import pandas as pd
from collections import defaultdict

file_path = dataset_mold.path / "credit.lisp"

with open(file_path, "r") as f:
    text = f.read()

# Storage
data = defaultdict(dict)
all_people = set()

# ----------------------------
# Helper: extract blocks
# ----------------------------
pred_blocks = re.findall(r"\(def-pred\s+([^\s]+).*?\)", text, re.DOTALL)

# Better block extraction
def extract_pred_block(name):
    pattern = rf"\(def-pred\s+{name}.*?\)\)"
    match = re.search(pattern, text, re.DOTALL)
    return match.group(0) if match else None

# ----------------------------
# 1. credit_screening
# ----------------------------
credit_block = extract_pred_block("credit_screening")

pos = re.findall(r":pos\s*\(\((.*?)\)\)", credit_block, re.DOTALL)
neg = re.findall(r":neg\s*\(\((.*?)\)\)", credit_block, re.DOTALL)

if pos:
    for s in re.findall(r"\(?(s\d+)\)?", pos[0]):
        data[s]["credit_screening"] = 1
        all_people.add(s)

if neg:
    for s in re.findall(r"\(?(s\d+)\)?", neg[0]):
        data[s]["credit_screening"] = 0
        all_people.add(s)

# ----------------------------
# 2. Unary predicates
# ----------------------------
unary_preds = [
    "jobless",
    "male",
    "female",
    "unmarried",
    "problematic_region"
]

for pred in unary_preds:
    block = extract_pred_block(pred)
    if block:
        matches = re.findall(r"\((s\d+)\)", block)
        for s in matches:
            data[s][pred] = 1
            all_people.add(s)

# ----------------------------
# 3. purchase_item (categorical)
# ----------------------------
block = extract_pred_block("purchase_item")
matches = re.findall(r"\((s\d+)\s+([^\s\)]+)\)", block)
for s, item in matches:
    data[s]["purchase_item"] = item
    all_people.add(s)

# ----------------------------
# 4. Numeric predicates
# ----------------------------
numeric_preds = [
    "age",
    "deposit",
    "monthly_payment",
    "numb_of_months",
    "numb_of_years_in_company"
]

for pred in numeric_preds:
    block = extract_pred_block(pred)
    if block:
        matches = re.findall(r"\((s\d+)\s+(\d+)\)", block)
        for s, val in matches:
            data[s][pred] = int(val)
            all_people.add(s)

# ----------------------------
# Build DataFrame
# ----------------------------
df = pd.DataFrame.from_dict(data, orient="index")

# Fill missing booleans with 0
for col in unary_preds:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

df = df.sort_index()

df.shape

(125, 12)

In [3]:
df["credit_screening"] = df["credit_screening"].fillna(0)

df["gender"] = None
df.loc[df["male"] == 1, "gender"] = "male"
df.loc[df["female"] == 1, "gender"] = "female"
df = df.drop(columns=["male", "female"])

as_cat_type = ["jobless", "unmarried", "problematic_region", "purchase_item", "gender"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 125
Columns: 11
Use sampling: False (sample size: 125)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['age', 'deposit', 'numb_of_years_in_company', 'monthly_payment', 'numb_of_months', 'purchase_item', 'unmarried', 'problematic_region', 'jobless', 'gender']
Rows remaining as candidates after top-10 filter: 0 (of 125)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,credit_screening,unmarried,purchase_item,age,deposit,monthly_payment,numb_of_months,numb_of_years_in_company,problematic_region,jobless,gender
0,0.0,0,bike,25,10,5,10,5.0,0,1,female
1,1.0,0,stereo,21,10,1,12,1.0,0,0,female
2,1.0,0,car,38,100,10,20,15.0,0,0,male
3,1.0,1,jewel,37,35,3,10,13.0,0,0,female
4,0.0,1,stereo,25,10,4,20,0.0,0,1,female


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,unmarried,category,0.0,0.0,2.0,"0, 1"
1,purchase_item,category,0.0,0.0,7.0,"jewel, stereo, medinstru, furniture, car, bike, pc"
2,problematic_region,category,0.0,0.0,2.0,"0, 1"
3,jobless,category,0.0,0.0,2.0,"0, 1"
4,gender,category,0.0,0.0,2.0,"male, female"
5,numb_of_years_in_company,float64,3.0,2.4,23.0,"1.0, 2.0, 0.0, 5.0, 3.0, 7.0, 13.0, 20.0, 15.0, 10.0"
6,credit_screening,float64,0.0,0.0,2.0,"1.0, 0.0"
7,age,int64,0.0,0.0,43.0,"25, 35, 18, 40, 50, 37, 23, 22, 27, 20"
8,deposit,int64,0.0,0.0,30.0,"10, 50, 5, 20, 100, 30, 150, 500, 200, 40"
9,monthly_payment,int64,0.0,0.0,20.0,"10, 5, 2, 4, 20, 3, 50, 7, 1, 15"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
credit_screening,125.0,0.680000,0.468353,0.0,1.0
age,125.0,36.720000,15.157560,18.0,98.0
deposit,125.0,69.464000,109.941006,1.0,500.0
monthly_payment,125.0,11.656000,14.348825,1.0,80.0
numb_of_months,125.0,13.144000,5.622111,4.0,30.0
numb_of_years_in_company,122.0,6.901639,8.086730,0.0,37.0


In [8]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column             rank                        
gender             1          male     65  52.0
                   2        female     60  48.0
jobless            1             0    111  88.8
                   2             1     14  11.2
problematic_region 1             0    111  88.8
                   2             1     14  11.2
purchase_item      1         jewel     32  25.6
                   2        stereo     27  21.6
                   3     medinstru     26  20.8
                   4     furniture     10   8.0
                   5           car     10   8.0
unmarried          1             0     66  52.8
                   2             1     59  47.2

In [9]:
# Target Distribution
target_df

,count,pct
credit_screening,,
1.0,85,68.0
0.0,40,32.0


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to japanese_credit_screening/019d736d-fca9-7bc3-aa72-3178801aca30


019d736d-fca9-7bc3-aa72-3178801aca30
21097808121e42b0bf64e13ca2b01fb3e032ec41bf72e272b056658c4427c523
